In [4]:
import random
import networkx as nx
import torch
import torch.nn as nn

from deep_ebm.utils_ebm import show_graph, evaluate_model
from deep_ebm.utils_ebm import save_graph, compare_graphs, show_graph_grid, compare_statistics
from deep_ebm.gnn_ebm import GraphDataset, GNN_EBM, train_one_epoch_pcd, gibbs_ministeps, evaluate_mmd_generated

from src.torch_erg import load_pglib_opf as lp
from src.torch_erg.samplers import GWGSampler, MHSampler, GWG_Hybrid_Sampler, MH_Hybrid_Sampler
from src.torch_erg.utils import laplacian_matrix,  unif_move, index_ravel_sampler
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle as pkl
from scipy.sparse.csgraph import connected_components
from scipy.sparse import csr_matrix

from Plots_and_utils.plots import *
from Plots_and_utils.other_g_stats import *

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


from src.torch_erg.utils import laplacian_matrix
from src.torch_erg.samplers import MHSampler_Hard

class Measurer(MHSampler_Hard):
    def __init__(self, backend: str):
        super().__init__(backend)

    def observables(self,mtx):
        L = laplacian_matrix(mtx)
        a3 = torch.matmul(torch.matmul(mtx,mtx),mtx)

        edges = torch.sum(mtx)/2
        triangles = torch.trace(a3)/6
        ac = torch.linalg.eigvalsh(L)[1]
        deg = torch.diagonal(L)
        tri_diag = torch.diagonal(a3) / 2

        valid = deg > 1
        local_clust = torch.zeros_like(deg)
        local_clust[valid] = tri_diag[valid] / (deg[valid] * (deg[valid] - 1))
        avg_clustering = torch.mean(local_clust[valid])

        avg_degree = torch.sum(mtx, dim=1).mean()

        return(torch.stack([edges, triangles, ac, avg_degree, avg_clustering]))

measurer = Measurer(backend='cpu')

Using device: cpu


In [5]:
with open("Saved_models/2community_GNN_EBM_50e_2000GS.pkl", "rb") as f:
    model_deep= pkl.load(f)

with open("data/TOY2community_dataset/TOY2comms_GraphDataset.pkl", "rb") as f:
    dataset_2community = pkl.load(f)

dataset_obs = [measurer.observables(g[0]) for g in dataset_2community]
torch.stack(dataset_obs).mean(axis = 0)

tensor([1.0603e+02, 1.1354e+02, 1.0209e-01, 7.0687e+00, 2.4433e-01])

In [6]:
reference = dataset_2community[0][0]
ref_feats = dataset_2community[0][1]
model_deep(ref_feats, reference)

tensor(188.6816, grad_fn=<SqueezeBackward0>)

In [7]:
class HybridSampler(GWG_Hybrid_Sampler):
    def observables(self, mtx):
        edges = torch.sum(mtx)/2
        triangles = torch.trace(torch.matmul(torch.matmul(mtx,mtx),mtx))/6
        return(torch.stack([edges, triangles]))

#we freeze the node features as we do not need them, so that we can evaluate the model only on the adj matrix
class NoFeatModel(nn.Module):
    def __init__(self, model, feats):
        super().__init__()
        self.model = model
        self.feats = feats.to(device)

    def forward(self,adj):
        return(self.model(self.feats, adj))

In [8]:
betas = torch.tensor([0., 0.])
weight_deep = 1
weight_erg = 1
niter_params = 500000
niter_sampling = 20000
params_update_every = 3
alpha = 0.001
min_change = 0.001

In [ ]:
beta_0 = [-0.5, -0.3, 0.1, 0.3, 0.5]
beta_1 = [-0.5, -0.3, 0.1, 0.3, 0.5]

adj_model = NoFeatModel(model_deep, ref_feats)

sampler = HybridSampler(backend="cpu",model = adj_model)
obs = sampler.observables(reference)
print("Reference observables: ", obs)
reference = reference.to(device)


Reference observables:  tensor([109., 127.])


In [10]:
n_beta0 = len(beta_0)
n_beta1 = len(beta_1)

means_array = [[0 for i in range(n_beta0)] for j in range(n_beta0)]
stds_array = [[0 for i in range(n_beta0)] for j in range(n_beta0)]

In [11]:
for i, b0 in enumerate(beta_0):
    for j, b1 in enumerate(beta_1):
        observables_deep, graphs_sampled = sampler.sample_run(
            graph=reference,
            observables=obs,
            params=torch.Tensor([b0, b1]),
            niter=3000,
            save_every=50,
            burn_in=0.1
        )
        
        sample_obs = [measurer.observables(g) for g in graphs_sampled]
        sample_obs_tensor = torch.stack(sample_obs)

        means_array[i][j] = sample_obs_tensor.mean(dim=0)
        stds_array[i][j]  = sample_obs_tensor.std(dim=0)
        

100%|██████████| 3000/3000 [00:11<00:00, 267.08it/s]


number of accepted steps is:  1334
number of rejected samples:  1666
Graph sampled:  61
-----------------------------------
Start obs:  tensor([109., 127.])
Mean obs:  tensor([58.1475, 13.5082])


100%|██████████| 3000/3000 [00:11<00:00, 260.56it/s]


number of accepted steps is:  1413
number of rejected samples:  1587
Graph sampled:  58
-----------------------------------
Start obs:  tensor([109., 127.])
Mean obs:  tensor([66.3621, 23.7414])


100%|██████████| 3000/3000 [00:11<00:00, 253.51it/s]


...

number of accepted steps is:  1179
number of rejected samples:  1821
Graph sampled:  62
-----------------------------------
Start obs:  tensor([109., 127.])
Mean obs:  tensor([176.2742, 486.9677])


100%|██████████| 3000/3000 [00:08<00:00, 374.64it/s]


number of accepted steps is:  508
number of rejected samples:  2492
Graph sampled:  61
-----------------------------------
Start obs:  tensor([109., 127.])
Mean obs:  tensor([ 247.7541, 1149.9508])


100%|██████████| 3000/3000 [00:07<00:00, 375.70it/s]

number of accepted steps is:  267
number of rejected samples:  2733
Graph sampled:  68
-----------------------------------
Start obs:  tensor([109., 127.])
Mean obs:  tensor([ 245.3088, 1170.8677])


In [12]:
means_array[0][0][0]

tensor(58.1475)

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Estrai il primo osservabile in un array numpy
z = np.array([[means_array[i][j][1].item() for j in range(n_beta1)] 
               for i in range(n_beta0)])

# Meshgrid per gli assi
B0, B1 = np.meshgrid(beta_1, beta_0)

# Crea la figura
fig = go.Figure(data=[
    go.Surface(
        x=B1,
        y=B0,
        z=z,
        colorscale="Viridis"
    )
])

# Etichette degli assi
fig.update_layout(
    title="Surface plot (observabile 0)",
    scene=dict(
        xaxis_title="beta1",
        yaxis_title="beta0",
        zaxis_title="Mean value of edges",
    ),
    width=800,
    height=600
)

fig.show()
